# 📈 Stock Forecasting Pipeline using LightGBM with Financial Report Injection

This pipeline predicts future stock prices for Israeli companies by combining **historical price-based technical indicators** with **quarterly financial report correlations**. It trains and saves a **LightGBM regression model per ticker**, and produces simulated 30-day forecasts based on model volatility.

---

## 💡 Key Components

- **Rolling Window Features**:
  - Rolling mean, std, and median over windows of 2, 5, 10, 20, 60 days.
- **Quarterly Financial Injection**:
  - Injected correlations from financial reports:
    - `Net Income`, `Revenue`, `Total Assets`
  - Captures:
    - Metric changes
    - Price change correlations (1d, 30d, 90d)
    - Score deltas (model-based score correlations)
- **Forward-Fill Strategy**:
  - Nonzero forward-fill of financial features for time continuity.

---

## ⚙️ Modeling Pipeline

- Standardizes all features using `StandardScaler`.
- Uses a **chronological train-test split** (no shuffling).
- Trains a separate **`LGBMRegressor`** per ticker:
  - 100 trees
  - Learning rate = 0.05
  - Max depth = 6
- Evaluates on unseen future data.

---

## 🔮 Forecasting Method

- Calculates daily percentage change volatility from model predictions.
- Simulates 30-day price forecasts using sampling from a fitted normal distribution (mean & std of % changes).

---

## 📤 Outputs

- 📁 **Model Checkpoints**: One `.pkl` file per trained stock model.
- 📄 **Metrics CSV**:
  - Evaluation scores
  - 30-day forecast per ticker
  - Raw predicted vs. actual y values

---

## 🧠 Evaluation Metrics

- **R-squared (R²)**: Goodness of fit
- **MAE**: Mean Absolute Error
- **MSE**: Mean Squared Error
- **MAPE**: Mean Absolute Percentage Error

---

## ✅ Robustness

- Handles missing tickers gracefully with try-except.
- Ensures all injected financial fields are initialized and forward-filled.
- Forecast logic adapts to final predicted value per model.



### Mounting Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Imports & Downloading data:

In [ ]:
# =========================
# 📆 LightGBM Pipeline with Report Parameter Injection
# =========================
import os
import json
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor
import pickle
import warnings
warnings.filterwarnings("ignore")

# =========================
# 📚 Helper Functions
# =========================
def generate_report_dates(year):
    return [
        pd.Timestamp(f"{year}-03-31"),
        pd.Timestamp(f"{year}-06-30"),
        pd.Timestamp(f"{year}-09-30"),
        pd.Timestamp(f"{year}-12-31")
    ]

# =========================
# 🔥 Load and Preprocess Stock Data
# =========================
df_tickers = pd.read_csv("/content/drive/Shareddrives/capstone project-stock market robo-advisor/symbols_Israel.csv")
tickers = df_tickers["0"].dropna().tolist()

# Download stock data
raw_data = yf.download(tickers, start="2020-01-01", end="2024-12-31", auto_adjust=False)['Close']

# Feature Engineering
for window in [2, 5, 10, 20, 60]:
    for ticker in tickers:
        raw_data[f'{ticker}_rolling_{window}'] = raw_data[ticker].rolling(window=window).mean()
        raw_data[f'{ticker}_rollingSTD_{window}'] = raw_data[ticker].rolling(window=window).std()
        raw_data[f'{ticker}_rollingMedian_{window}'] = raw_data[ticker].rolling(window=window).median()

raw_data = raw_data.fillna(method='ffill').fillna(method='bfill')

# =========================
# 📊 Initialize Outputs
# =========================
metrics_df = pd.DataFrame()
save_dir = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/LightGBM models with report parameters"
os.makedirs(save_dir, exist_ok=True)

[**********************69%********               ]  350 of 510 completedERROR:yfinance:Could not get exchangeTimezoneName for ticker '???.TA' reason: 'chart'
[*********************100%***********************]  510 of 510 completed
ERROR:yfinance:
15 Failed downloads:
ERROR:yfinance:['LVPR.TA', 'INFR-M.TA', 'UNCR.TA', 'ECPA-M.TA', 'AICS-M.TA', 'PMCN.TA', 'CNZN.TA', 'ARAD.TA', 'YAAC.TA', 'ENDY.TA', 'GLTC.TA', '???.TA', 'HDHA.TA', 'UNVO.TA', 'INTO.TA']: YFTzMissingError('possibly delisted; no timezone found')


### Models Building:

In [ ]:

# =========================
# ⏳ Loop Over Tickers
# =========================
for ticker in tickers:
    try:
        print(f"\n================ Processing {ticker} ================")

        # -------------------------
        # 🌐 Financial Parameters Injection
        # -------------------------
        metrics_features = ['Net Income', 'Total Assets', 'Revenue']
        financial_dates = []
        for y in range(2020, 2025):
            financial_dates.extend(generate_report_dates(y))

        financial_features = pd.DataFrame(0.0, index=raw_data.index, columns=[
            f'{feature}_metrics_change' for feature in metrics_features] +
            [f'{feature}_price_change_last_day' for feature in metrics_features] +
            [f'{feature}_price_change_30d' for feature in metrics_features] +
            [f'{feature}_price_change_90d' for feature in metrics_features] +
            [f'{feature}_score_last_day' for feature in metrics_features] +
            [f'{feature}_score_30d' for feature in metrics_features] +
            [f'{feature}_score_90d' for feature in metrics_features]
        )

        for date in pd.to_datetime(financial_dates):
            try:
                year = date.year if date.month != 3 or date.year == 2020 else date.year - 1
                quarter = ('Q1' if date.month == 3 else 'Q2' if date.month == 6 else 'Q3' if date.month == 9 else 'Q4')
                next_quarter = ('Q2' if quarter == 'Q1' else 'Q3' if quarter == 'Q2' else 'Q4' if quarter == 'Q3' else 'Q1')
                next_year = year if quarter != 'Q4' else year + 1
                if next_year > 2024:
                    continue

                base_path = "/content/drive/Shareddrives/capstone project-stock market robo-advisor/Reports Parameters/Correlations"
                metrics_path = f"{base_path}/correlation metrics change/{quarter}_{year}_to_{next_quarter}_{next_year}.csv"
                price_change_path = f"{base_path}/correlations price change/{quarter}_{year}_to_{next_quarter}_{next_year}.csv"
                score_path = f"{base_path}/correlation score/{quarter}_{year}_to_{next_quarter}_{next_year}.csv"

                cmc = pd.read_csv(metrics_path, index_col=0)
                cpc = pd.read_csv(price_change_path, index_col=0)
                cs = pd.read_csv(score_path, index_col=0)

                if date not in financial_features.index:
                    continue

                for feature in metrics_features:
                    if ticker in cmc.index:
                        val = cmc.loc[ticker, feature]
                        val_list_price = eval(cpc.loc[ticker, feature])
                        val_list_score = eval(cs.loc[ticker, feature])
                    else:
                        val = 0
                        val_list_price = [0, 0, 0]
                        val_list_score = [0, 0, 0]

                    financial_features.at[date, f'{feature}_metrics_change'] = val
                    financial_features.at[date, f'{feature}_price_change_last_day'] = val_list_price[0]
                    financial_features.at[date, f'{feature}_price_change_30d'] = val_list_price[1]
                    financial_features.at[date, f'{feature}_price_change_90d'] = val_list_price[2]
                    financial_features.at[date, f'{feature}_score_last_day'] = val_list_score[0]
                    financial_features.at[date, f'{feature}_score_30d'] = val_list_score[1]
                    financial_features.at[date, f'{feature}_score_90d'] = val_list_score[2]

            except Exception as e:
                print(f"Skipping financials for {date.date()}: {e}")

        def ffill_nonzero(df):
            for col in df.columns:
                last_val = None
                for idx in df.index:
                    if df.at[idx, col] != 0.0:
                        last_val = df.at[idx, col]
                    elif last_val is not None:
                        df.at[idx, col] = last_val
            return df

        financial_features = ffill_nonzero(financial_features)
        raw_data = raw_data.dropna(axis=1, how='all')

        # -------------------------
        # 🔹 Combine All Features
        # -------------------------
        full_df = pd.concat([raw_data, financial_features], axis=1).dropna()
        if ticker not in full_df.columns:
            continue

        y = full_df[ticker]
        X = full_df.drop(columns=[ticker])

        scaler = StandardScaler()
        X_scaled = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)

        # Chronological train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.1, shuffle=False
        )

        # Train model
        model = LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=42)
        model.fit(X_train, y_train)

        # Evaluate
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)

        # Forecast future prices
        future_pred = [y_pred[-1]]
        for _ in range(30):
            pct_changes = pd.Series(y_pred).pct_change().dropna()
            mean_pct = pct_changes.mean()
            std_pct = pct_changes.std()
            future_pred.append(future_pred[-1] * (1 + np.random.normal(mean_pct, std_pct)))
            y_pred.append(future_pred[-1])
        y_pred = y_pred[:-30]
        future_pred = future_pred[1:]

        forecast_columns = {f"Day_{i+1}": future_pred[i] for i in range(30)}
        metrics_data = {
            "Stock": ticker,
            "R-squared": r2,
            "MAE": mae,
            "MSE": mse,
            "MAPE": mape,
            **forecast_columns,
            "y_test": json.dumps(y_test.tolist()),
            "y_pred": json.dumps(y_pred.tolist())
        }
        metrics_df = pd.concat([metrics_df, pd.DataFrame([metrics_data])], ignore_index=True)

        # Save model
        model_path = os.path.join(save_dir, f"{ticker.replace('.', '_')}_LightGBM_model.pkl")
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)

        print(f"✅ {ticker} saved. R2: {r2:.4f}, MAE: {mae:.2f}, MAPE: {mape:.2f}, MSE: {mse:.2f}")

    except Exception as e:
        print(f"❌ Failed {ticker}: {e}")

# Save final CSV
metrics_df.to_csv(os.path.join(save_dir, "metrics_lightgbm_with_report_parameters_from_400.csv"), index=False)
print("\n📅 All stocks processed.")


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
✅ STEC.TA saved. R2: 0.5432, MAE: 43.84, MAPE: 2.23, MSE: 10338.51

================ Processing SNEL.TA ================
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.246629 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1808844
[LightGBM] [Info] Number of data points in the train set: 983, number of used features: 7829
[Light